## Ex:2 Data Wrangling and Transformation 
### Objective

To perform data wrangling and transformation on a dataset using Python and Pandas by handling missing values, removing duplicates, correcting data types, filtering data, and transforming variables into a suitable format for data analysis and machine learning.



##  Dataset Description

The dataset contains information about **student academic performance, educational background, MBA specialization, and placement salary**.

###  Dataset Attributes

| Column | Description |
|:---|:---|
| `sl_no` | Serial number / student identifier |
| `gender` | Gender of the student |
| `hsc_p` | Higher Secondary / 12th percentage |
| `hsc_s` | Higher Secondary stream |
| `degree_p` | Undergraduate degree percentage |
| `degree_t` | Undergraduate degree type |
| `etest_p` | Employability / entrance test percentage |
| `specialisation` | MBA specialization |
| `mba_p` | MBA percentage |
| `salary` | Salary offered after placement |

---

## Experiment Question

 **Using the given student placement dataset, perform data wrangling and transformation by handling missing values, scaling numerical features, detecting and treating outliers, encoding categorical variables, and generating a final model-ready dataset.**




Data wrangling and transformation is the process of converting raw student placement data into a clean, consistent, and machine-learning-ready dataset. In this experiment, the given student placement dataset is first loaded into a Pandas DataFrame and explored by examining its rows, columns, data types, and descriptive statistics. Missing values are then identified and handled by removing records with missing `salary` values and replacing missing values in `hsc_p`, `degree_p`, and `etest_p` with their respective mean values. The numerical attributes such as `hsc_p`, `degree_p`, `etest_p`, and `salary` are transformed using feature-scaling techniques such as `StandardScaler` and `MinMaxScaler` to bring the variables into suitable numerical ranges. The dataset is then divided into input features (`X`) and the target variable (`Y`), where `salary` is considered the target. The preprocessed data is saved as `Pre.csv` for further processing. Next, possible outliers in the `salary` attribute are identified using a boxplot and statistically detected using the Z-score method, where values with an absolute Z-score greater than 3 are considered potential outliers. Outliers are also treated using the capping and flooring method by calculating the 5th and 95th percentiles and replacing values outside these limits with the corresponding boundary values. The salary distribution before and after outlier treatment is then compared using visualization. Since the dataset also contains categorical attributes such as `gender`, `hsc_s`, `degree_t`, and `specialisation`, these variables are converted into numerical representations using `LabelEncoder` and One-Hot Encoding. Finally, the completely transformed dataset is verified for missing values, data types, dimensions, and numerical representation, and the resulting model-ready dataset is saved as `Final.csv`. Thus, the experiment demonstrates the complete workflow of preparing real-world tabular data for data analysis and machine-learning applications.

## Step 1: Start by importing the necessary Python libraries for data preprocessing.


In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.preprocessing import LabelEncoder
from scipy.stats import zscore
from scipy import stats
from sklearn.preprocessing import LabelEncoder

In [1]:
%pip install scikit-learn scipy

Note: you may need to restart the kernel to use updated packages.


## Step 2: Load the placement dataset into a Pandas Dataframe.

In [24]:
df=pd.read_csv("data.csv")
df.info()
df.shape 
df.head 
df.tail
df.sample(5)
df.describe()
df.loc
df.iloc[0]
df[0:2]
df["degree_p"]

<class 'pandas.DataFrame'>
RangeIndex: 215 entries, 0 to 214
Data columns (total 10 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   sl_no           215 non-null    int64  
 1   gender          215 non-null    str    
 2   hsc_p           210 non-null    float64
 3   hsc_s           215 non-null    str    
 4   degree_p        213 non-null    float64
 5   degree_t        215 non-null    str    
 6   etest_p         211 non-null    float64
 7   specialisation  215 non-null    str    
 8   mba_p           214 non-null    float64
 9   salary          148 non-null    float64
dtypes: float64(5), int64(1), str(4)
memory usage: 16.9 KB


0      58.00
1      77.48
2      64.00
3        NaN
4      73.30
       ...  
210    77.60
211    72.00
212    73.00
213    58.00
214    53.00
Name: degree_p, Length: 215, dtype: float64

Step 3:Take a quick look at the data to understand its structure and identify any missing values or anomalies.

In [25]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 215 entries, 0 to 214
Data columns (total 10 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   sl_no           215 non-null    int64  
 1   gender          215 non-null    str    
 2   hsc_p           210 non-null    float64
 3   hsc_s           215 non-null    str    
 4   degree_p        213 non-null    float64
 5   degree_t        215 non-null    str    
 6   etest_p         211 non-null    float64
 7   specialisation  215 non-null    str    
 8   mba_p           214 non-null    float64
 9   salary          148 non-null    float64
dtypes: float64(5), int64(1), str(4)
memory usage: 16.9 KB


#### The method isnull() checks each element in the DataFrame (or Series) to see if it is NaN (Not a Number) or None (missing value).
It returns a DataFrame (or Series) of the same shape as the input, with Boolean values:
#### True: The value is null (NaN or None).
#### False: The value is not null.

In [26]:
df.isnull().sum()

sl_no              0
gender             0
hsc_p              5
hsc_s              0
degree_p           2
degree_t           0
etest_p            4
specialisation     0
mba_p              1
salary            67
dtype: int64

## Step 4: Handle Missing Data
### Option 1: If the dataset is large and only a small percentage of data is missing, you can remove rows with missing values using dropna(subset,inplace)


In [4]:
df.dropna(subset=["salary"],inplace=True)
df.isnull().sum()

sl_no             0
gender            0
hsc_p             2
hsc_s             0
degree_p          1
degree_t          0
etest_p           2
specialisation    0
mba_p             0
salary            0
dtype: int64

### Option 2:If removing data isn't ideal, you can impute (df.[""].fillna(df[""].mean(),inplace)) missing values using methods like mean, median, or most frequent.

In [5]:
df["hsc_p"]=df["hsc_p"].fillna(df["hsc_p"].mean())
df["degree_p"]=df["degree_p"].fillna(df["degree_p"].mean())
df["etest_p"]=df["etest_p"].fillna(df["etest_p"].mean())
df.isnull().sum()

sl_no             0
gender            0
hsc_p             0
hsc_s             0
degree_p          0
degree_t          0
etest_p           0
specialisation    0
mba_p             0
salary            0
dtype: int64

## Step 5: Feature Scaling
Feature scaling is the process of converting numerical features to a similar scale so that one feature does not dominate another simply because it has larger numerical values.

<img src="https://i.postimg.cc/G21gMYnF/f.png" alt="Image Description" width="500">









## Option 1( StandardScaler): This method scales the data to have a mean of 0 and a standard deviation of 1.


In [6]:
c=["hsc_p","degree_p","etest_p","salary"]
s1=StandardScaler()
df[c]=s1.fit_transform(df[c])
df.head()

,sl_no,gender,hsc_p,hsc_s,degree_p,degree_t,etest_p,specialisation,mba_p,salary
0,1,M,2.265997e+00,Commerce,-1.652293,Sci&Tech,-1.328518,Mkt&HR,58.80,-0.200292
1,2,M,8.987875e-01,Science,1.346845,Sci&Tech,0.978332,Mkt&Fin,66.28,-0.951839
2,3,M,1.533482e-15,Arts,-0.728534,Comm&Mgmt,0.136149,Mkt&Fin,57.80,-0.415019
4,5,M,3.883770e-01,Commerce,0.703293,Comm&Mgmt,1.732635,Mkt&Fin,55.50,1.463849
7,8,M,-6.475513e-01,Science,-0.420614,Sci&Tech,-0.449718,Mkt&Fin,62.14,-0.393547


#### Option 2:This method scales the data to a fixed range, usually between 0 and 1. 
###  MinMaxScaler()

In [8]:
from sklearn.preprocessing import MinMaxScaler

s2 = MinMaxScaler()

df[c] = s2.fit_transform(df[c])

df.head()

,sl_no,gender,hsc_p,hsc_s,degree_p,degree_t,etest_p,specialisation,mba_p,salary
0,1,M,0.857051,Commerce,0.057143,Sci&Tech,0.104167,Mkt&HR,58.80,0.094595
1,2,M,0.586729,Science,0.613714,Sci&Tech,0.760417,Mkt&Fin,66.28,0.000000
2,3,M,0.409023,Arts,0.228571,Comm&Mgmt,0.520833,Mkt&Fin,57.80,0.067568
4,5,M,0.485812,Commerce,0.494286,Comm&Mgmt,0.975000,Mkt&Fin,55.50,0.304054
7,8,M,0.280990,Science,0.285714,Sci&Tech,0.354167,Mkt&Fin,62.14,0.070270


## Step 6  Option 1: Identifying Outliers Using Z-Scores
The value of 3 in the context of Z-scores is often used as a threshold to identify outliers in a dataset. A Z-score represents how many standard deviations a data point is away from the mean of the dataset. Specifically:

A Z-score of 0 means the data point is exactly at the mean.
A Z-score of 1 means the data point is one standard deviation above the mean, and so on.
A Z-score of 3 corresponds to a data point being 3 standard deviations away from the mean. For a normal distribution, about 99.7% of the data points fall within 3 standard deviations of the mean (according to the 68-95-99.7 rule, which describes the spread of data in a normal distribution). Therefore, points with Z-scores greater than 3 or less than -3 are considered unusually far from the mean and are often flagged as outliers.

This threshold (Z > 3 or Z < -3) is commonly used in many statistical applications because it captures the extreme values that are rare in a normal distribution, which are typically considered to be outliers. However, the choice of threshold can vary depending on the specific application and the nature of the data.



[![Chat-GPT-Image-Aug-20-2026-08-06-05-PM.png](https://i.postimg.cc/2SFTtcXL/Chat-GPT-Image-Aug-20-2026-08-06-05-PM.png)](https://postimg.cc/4Yyz75GX)

In [11]:
from scipy import stats

columns_to_check = ["salary"]

z_score = stats.zscore(df[columns_to_check])

u = (z_score > 3)
l = (z_score < -3)

indx = u | l

clean_df = df[~indx[:, 0]]

df.info()
clean_df.info()

<class 'pandas.DataFrame'>
Index: 148 entries, 0 to 213
Data columns (total 10 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   sl_no           148 non-null    int64  
 1   gender          148 non-null    str    
 2   hsc_p           148 non-null    float64
 3   hsc_s           148 non-null    str    
 4   degree_p        148 non-null    float64
 5   degree_t        148 non-null    str    
 6   etest_p         148 non-null    float64
 7   specialisation  148 non-null    str    
 8   mba_p           148 non-null    float64
 9   salary          148 non-null    float64
dtypes: float64(5), int64(1), str(4)
memory usage: 12.7 KB
<class 'pandas.DataFrame'>
Index: 145 entries, 0 to 213
Data columns (total 10 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   sl_no           145 non-null    int64  
 1   gender          145 non-null    str    
 2   hsc_p           145 non-null    float6

### Option 2:  Capping and Flooring Outliers
Capping and flooring is an outlier-treatment technique where extreme values are replaced with predefined boundary values instead of deleting the records.

In [12]:
l1=df["salary"].quantile(0.05)
u1=df["salary"].quantile(0.95)
df_capped=df.copy()
df_capped["salary"]=df_capped["salary"].clip(l1,u1)
df_capped.info()
df.head()


<class 'pandas.DataFrame'>
Index: 148 entries, 0 to 213
Data columns (total 10 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   sl_no           148 non-null    int64  
 1   gender          148 non-null    str    
 2   hsc_p           148 non-null    float64
 3   hsc_s           148 non-null    str    
 4   degree_p        148 non-null    float64
 5   degree_t        148 non-null    str    
 6   etest_p         148 non-null    float64
 7   specialisation  148 non-null    str    
 8   mba_p           148 non-null    float64
 9   salary          148 non-null    float64
dtypes: float64(5), int64(1), str(4)
memory usage: 12.7 KB


,sl_no,gender,hsc_p,hsc_s,degree_p,degree_t,etest_p,specialisation,mba_p,salary
0,1,M,0.857051,Commerce,0.057143,Sci&Tech,0.104167,Mkt&HR,58.80,0.094595
1,2,M,0.586729,Science,0.613714,Sci&Tech,0.760417,Mkt&Fin,66.28,0.000000
2,3,M,0.409023,Arts,0.228571,Comm&Mgmt,0.520833,Mkt&Fin,57.80,0.067568
4,5,M,0.485812,Commerce,0.494286,Comm&Mgmt,0.975000,Mkt&Fin,55.50,0.304054
7,8,M,0.280990,Science,0.285714,Sci&Tech,0.354167,Mkt&Fin,62.14,0.070270


## Step 7: Convert categorical variables into numerical format using LabelEncoder ().
[![Picture1.png](https://i.postimg.cc/yNpNvnVd/Picture1.png)](https://postimg.cc/zLW5fCHZ)




In [6]:
import os

print(os.getcwd())

C:\Users\ISHA MANOJ\Downloads\Lab-2


In [7]:
print(os.listdir())

['.ipynb_checkpoints', 'Automobile.csv', 'data.csv', 'Ex2.ipynb']


In [ ]:
Convert categorical variables into numerical format using one hot encoder
Picture2.png

In [27]:
L1=LabelEncoder()
df["gender"]=L1.fit_transform(df["gender"])
df["degree_t"]=L1.fit_transform(df["degree_t"])
df["specialisation"]=L1.fit_transform(df["specialisation"])
df.head()

,sl_no,gender,hsc_p,hsc_s,degree_p,degree_t,etest_p,specialisation,mba_p,salary
0,1,1,91.00,Commerce,58.00,2,55.0,1,58.80,270000.0
1,2,1,78.33,Science,77.48,2,86.5,0,66.28,200000.0
2,3,1,NaN,Arts,64.00,0,75.0,0,57.80,250000.0
3,4,1,52.00,Science,NaN,2,66.0,1,59.43,NaN
4,5,1,73.60,Commerce,73.30,0,96.8,0,55.50,425000.0


In [ ]:
## Convert categorical variables into numerical format using one hot encoder



In [17]:
import pandas as pd

df = pd.read_csv("data.csv")



print(df.columns)

Index(['sl_no', 'gender', 'hsc_p', 'hsc_s', 'degree_p', 'degree_t', 'etest_p',
       'specialisation', 'mba_p', 'salary'],
      dtype='str')


In [18]:
c = ["gender"]

one_hot_encoded_data = pd.get_dummies(df, columns=c)

one_hot_encoded_data.to_csv("clean.csv", index=False)

one_hot_encoded_data.head()

,sl_no,hsc_p,hsc_s,degree_p,degree_t,etest_p,specialisation,mba_p,salary,gender_F,gender_M
0,1,91.00,Commerce,58.00,Sci&Tech,55.0,Mkt&HR,58.80,270000.0,False,True
1,2,78.33,Science,77.48,Sci&Tech,86.5,Mkt&Fin,66.28,200000.0,False,True
2,3,NaN,Arts,64.00,Comm&Mgmt,75.0,Mkt&Fin,57.80,250000.0,False,True
3,4,52.00,Science,NaN,Sci&Tech,66.0,Mkt&HR,59.43,NaN,False,True
4,5,73.60,Commerce,73.30,Comm&Mgmt,96.8,Mkt&Fin,55.50,425000.0,False,True


In [19]:
c=["gender"]
one_hot_encoded_data=pd.get_dummies(df,columns=c)
one_hot_encoded_data.to_csv('clean.csv', index=False)
one_hot_encoded_data.head()
Y=df['salary'].copy()
X=df.drop(columns=['salary','sl_no']).copy()

# Exercise: Data Cleaning and Transformation – Automobile Dataset

## Step 1: Load and Explore the Dataset

### 1. Load the Dataset
- Import Pandas and load the Automobile dataset.
- Display the first 10 rows.
- Display the shape of the dataset.

### 2. Explore the Dataset
- Display the column names.
- Display the data types.
- Generate descriptive statistics.
- Identify numerical and categorical columns.
- Display unique values in categorical columns.

## Step 2: Data Cleaning

### 3. Check Missing Values
- Check for missing values in each column.
- Display the number and percentage of missing values.

### 4. Handle Missing Values
- Replace missing numerical values using mean or median.
- Replace missing categorical values using mode.
- Verify that no missing values remain.

### 5. Remove Duplicate Records
- Check for duplicate rows.
- Display the number of duplicate records.
- Remove duplicate records.
- Verify the result.

### 6. Clean the `horsepower` Column
- Identify non-numeric values such as `?`.
- Replace `?` with `NaN`.
- Convert `horsepower` to numeric.
- Handle the resulting missing values.

## Step 3: Data Transformation

### 7. Transform the `origin` Column
- Display the unique values in `origin`.
- Convert the values into meaningful labels:
  - `1` → `usa`
  - `2` → `europe`
  - `3` → `japan`

### 8. Create `weight_kg`
- Create a new column `weight_kg`.
- Convert weight from pounds to kilograms.

  `weight_kg = weight × 0.453592`

### 9. Create `mpg_category`
Create a new column based on `mpg`:
- `< 20` → `Low`
- `20–29` → `Medium`
- `≥ 30` → `High`

### 10. Create `vehicle_age`
- Create a new column `vehicle_age`.
- Assume the current year is 2026.

  `vehicle_age = 2026 - model_year`

### 11. Rename Columns
Rename:
- `mpg` → `miles_per_gallon`
- `horsepower` → `hp`
- `weight` → `weight_lbs`
- `model_year` → `year`

### 12. Filter the Data
Display vehicles:
- With `mpg > 30`
- With `horsepower > 150`
- With `cylinders >= 6`
- Manufactured after 1980
- Originating from `usa`

## Step 4: Encoding Categorical Data

### 13. Label Encoding
- Apply `LabelEncoder` to the `origin` column.
- Create a new column `origin_encoded`.
- Display the original and encoded values.
- Display the category-to-label mapping.

### 14. One-Hot Encoding
- Apply One-Hot Encoding to the `origin` column.
- Compare Label Encoding and One-Hot Encoding.
- Which encoding method is more appropriate for `origin`? Explain why.

## Step 5: Outlier Detection

### 15. Identify Outliers Using Z-Scores
- Calculate the Z-score for the numerical features.
- Identify observations with `|Z-score| > 3` as outliers.
- Count the outliers in each numerical column.
- Display the rows containing outliers.
- Decide whether the outliers should be removed or retained.

## Step 6: Feature Scaling

### 16. Standardization Using StandardScaler
- Select the numerical features.
- Apply `StandardScaler`.
- Display the standardized values.
- Verify that the features have approximately mean `0` and standard deviation `1`.

## Step 7: Normalization

### 17. Normalization Using MinMaxScaler
- Apply `MinMaxScaler` to the numerical features.
- Transform the features to the range `[0, 1]`.
- Display the normalized values.
- Compare **Standardization** and **Normalization**.
- Explain when each scaling method is appropriate.

## Step 8: Create Features and Target

### 18. Create X and Y Variables
- Select the appropriate input features as **X (independent variables)**.
- Select `mpg` as **Y (target variable)**.
- Display the shape of `X` and `Y`.
- Save `X` and `Y` into `automobile_X_Y.csv`.
- Load the CSV file again and display the first 5 rows.

## Step 9: Save the Final Dataset

### 19. Save the Preprocessed Dataset
- Combine the processed features and target variable.
- Display the final dataset.
- Check for missing values.
- Save the final dataset as `automobile_preprocessed.csv`.

In [5]:
#STEP 1 — LOAD AND EXPLORE THE DATASET
#1. Load the Dataset

import pandas as pd
import numpy as np

df = pd.read_csv("Automobile.csv")

print("First 10 rows:")
display(df.head(10))

print("Shape of the dataset:")
print(df.shape)

First 10 rows:


,name,mpg,cylinders,displacement,horsepower,weight,acceleration,model_year,origin
0,chevrolet chevelle malibu,18.0,8.0,307.0,130.0,3504.0,12.0,70,usa
1,buick skylark 320,15.0,8.0,350.0,165.0,3693.0,11.5,70,usa
2,plymouth satellite,18.0,8.0,318.0,150.0,3436.0,11.0,70,usa
3,amc rebel sst,16.0,8.0,304.0,150.0,3433.0,12.0,70,usa
4,ford torino,17.0,NaN,302.0,140.0,3449.0,10.5,70,usa
5,ford galaxie 500,15.0,8.0,429.0,198.0,4341.0,10.0,70,usa
6,chevrolet impala,14.0,8.0,454.0,220.0,NaN,9.0,70,usa
7,plymouth fury iii,14.0,8.0,440.0,215.0,4312.0,8.5,70,usa
8,pontiac catalina,14.0,8.0,455.0,NaN,4425.0,10.0,70,usa
9,amc ambassador dpl,15.0,8.0,390.0,190.0,3850.0,8.5,70,usa


Shape of the dataset:
(398, 9)


In [2]:
#2. Explore the Dataset

print("Column names:")
print(df.columns.tolist())

print("\nData types:")
print(df.dtypes)

print("\nDescriptive statistics:")
display(df.describe())

# Numerical columns
numerical_columns = df.select_dtypes(include=np.number).columns.tolist()

# Categorical columns
categorical_columns = df.select_dtypes(exclude=np.number).columns.tolist()

print("\nNumerical columns:")
print(numerical_columns)

print("\nCategorical columns:")
print(categorical_columns)

Column names:
['name', 'mpg', 'cylinders', 'displacement', 'horsepower', 'weight', 'acceleration', 'model_year', 'origin']

Data types:
name                str
mpg             float64
cylinders       float64
displacement    float64
horsepower      float64
weight          float64
acceleration    float64
model_year        int64
origin              str
dtype: object

Descriptive statistics:


,mpg,cylinders,displacement,horsepower,weight,acceleration,model_year
count,398.000000,395.000000,395.000000,386.000000,396.000000,395.000000,398.000000
mean,23.514573,5.445570,193.340506,104.316062,2965.025253,15.562278,76.010050
std,7.815984,1.696203,104.425993,38.086281,845.254458,2.750260,3.697627
min,9.000000,3.000000,68.000000,46.000000,1613.000000,8.000000,70.000000
25%,17.500000,4.000000,102.500000,75.250000,2222.250000,13.850000,73.000000
50%,23.000000,4.000000,146.000000,92.500000,2797.500000,15.500000,76.000000
75%,29.000000,8.000000,262.000000,125.000000,3581.750000,17.150000,79.000000
max,46.600000,8.000000,455.000000,230.000000,5140.000000,24.800000,82.000000



Numerical columns:
['mpg', 'cylinders', 'displacement', 'horsepower', 'weight', 'acceleration', 'model_year']

Categorical columns:
['name', 'origin']


In [3]:
#Display unique values in categorical columns

for col in categorical_columns:
    print(f"\nUnique values in {col}:")
    print(df[col].unique())


Unique values in name:
<StringArray>
[ 'chevrolet chevelle malibu',          'buick skylark 320',
         'plymouth satellite',              'amc rebel sst',
                'ford torino',           'ford galaxie 500',
           'chevrolet impala',          'plymouth fury iii',
           'pontiac catalina',         'amc ambassador dpl',
 ...
 'chrysler lebaron medallion',             'ford granada l',
           'toyota celica gt',          'dodge charger 2.2',
           'chevrolet camaro',            'ford mustang gl',
                  'vw pickup',              'dodge rampage',
                'ford ranger',                 'chevy s-10']
Length: 305, dtype: str

Unique values in origin:
<StringArray>
['usa', 'japan', 'europe']
Length: 3, dtype: str


In [4]:
#STEP 2 — DATA CLEANING
#3. Check Missing Values

missing_count = df.isnull().sum()

missing_percentage = (df.isnull().sum() / len(df)) * 100

missing_table = pd.DataFrame({
    "Missing Count": missing_count,
    "Missing Percentage": missing_percentage
})

display(missing_table)

,Missing Count,Missing Percentage
name,0,0.000000
mpg,0,0.000000
cylinders,3,0.753769
displacement,3,0.753769
horsepower,12,3.015075
weight,2,0.502513
acceleration,3,0.753769
model_year,0,0.000000
origin,0,0.000000


In [5]:
#4. Handle Missing Values

df["horsepower"] = df["horsepower"].replace("?", np.nan)
df["horsepower"] = pd.to_numeric(df["horsepower"], errors="coerce")
numerical_columns = df.select_dtypes(include=np.number).columns

for col in numerical_columns:
    df[col] = df[col].fillna(df[col].median())
    categorical_columns = df.select_dtypes(exclude=np.number).columns

for col in categorical_columns:
    if df[col].isnull().sum() > 0:
        df[col] = df[col].fillna(df[col].mode()[0])
print("Missing values after cleaning:")
print(df.isnull().sum())

Missing values after cleaning:
name            0
mpg             0
cylinders       0
displacement    0
horsepower      0
weight          0
acceleration    0
model_year      0
origin          0
dtype: int64


In [6]:
#5. Remove Duplicate Records

duplicate_count = df.duplicated().sum()
print("Number of duplicate records:", duplicate_count)
df = df.drop_duplicates()
print("Number of duplicate records after removal:", df.duplicated().sum())
print("Shape after removing duplicates:", df.shape)

Number of duplicate records: 0
Number of duplicate records after removal: 0
Shape after removing duplicates: (398, 9)


In [7]:
#6. Clean the horsepower Column

print("Unique horsepower values before cleaning:")
print(df["horsepower"].unique())
df["horsepower"] = df["horsepower"].replace("?", np.nan)
df["horsepower"] = pd.to_numeric(df["horsepower"], errors="coerce")
df["horsepower"] = df["horsepower"].fillna(df["horsepower"].median())
print("Horsepower data type:", df["horsepower"].dtype)
print("Missing horsepower values:", df["horsepower"].isnull().sum())

Unique horsepower values before cleaning:
[130.  165.  150.  140.  198.  220.  215.   92.5 190.  170.  160.  225.
  95.   97.   85.   88.   87.   90.  113.  200.  210.  193.  100.  105.
 175.  153.  180.  110.   72.   86.   70.   76.   65.   69.   60.   80.
  54.  208.  155.  112.   92.  145.  137.  158.   46.  167.   94.  107.
 230.   49.   75.   91.  122.   67.   83.   78.   52.   61.   93.  148.
 129.   96.   71.   98.  115.   53.   81.   79.  120.  152.  102.  108.
  68.   58.  149.   89.   63.   48.   66.  139.  103.  125.  133.  138.
 135.  142.   77.   62.  132.   84.   64.   74.  116.   82. ]
Horsepower data type: float64
Missing horsepower values: 0


In [8]:
#STEP 3 — DATA TRANSFORMATION
#7. Transform the origin Column

print("Unique origin values:")
print(df["origin"].unique())

origin_mapping = {
    1: "usa",
    2: "europe",
    3: "japan"
}

df["origin"] = df["origin"].map(origin_mapping)

print("Origin values after transformation:")
print(df["origin"].unique())

display(df[["origin"]].head(10))

Unique origin values:
<StringArray>
['usa', 'japan', 'europe']
Length: 3, dtype: str
Origin values after transformation:
<StringArray>
[nan]
Length: 1, dtype: str


,origin
0,NaN
1,NaN
2,NaN
3,NaN
4,NaN
5,NaN
6,NaN
7,NaN
8,NaN
9,NaN


In [9]:
#8. Create weight_kg

df["weight_kg"] = df["weight"] * 0.453592

display(df[["weight", "weight_kg"]].head(10))

,weight,weight_kg
0,3504.0,1589.386368
1,3693.0,1675.115256
2,3436.0,1558.542112
3,3433.0,1557.181336
4,3449.0,1564.438808
5,4341.0,1969.042872
6,2797.5,1268.923620
7,4312.0,1955.888704
8,4425.0,2007.144600
9,3850.0,1746.329200


In [10]:
#9. Create mpg_category

def mpg_category(mpg):
    if mpg < 20:
        return "Low"
    elif mpg < 30:
        return "Medium"
    else:
        return "High"

df["mpg_category"] = df["mpg"].apply(mpg_category)

display(df[["mpg", "mpg_category"]].head(10))

print(df["mpg_category"].value_counts())

,mpg,mpg_category
0,18.0,Low
1,15.0,Low
2,18.0,Low
3,16.0,Low
4,17.0,Low
5,15.0,Low
6,14.0,Low
7,14.0,Low
8,14.0,Low
9,15.0,Low


mpg_category
Medium    155
Low       151
High       92
Name: count, dtype: int64


In [11]:
#10. Create vehicle_age

df["vehicle_age"] = 2026 - df["model_year"]

display(df[["model_year", "vehicle_age"]].head(10))

,model_year,vehicle_age
0,70,1956
1,70,1956
2,70,1956
3,70,1956
4,70,1956
5,70,1956
6,70,1956
7,70,1956
8,70,1956
9,70,1956


In [12]:
#11. Rename Columns

df.rename(columns={
    "mpg": "miles_per_gallon",
    "horsepower": "hp",
    "weight": "weight_lbs",
    "model_year": "year"
}, inplace=True)

print("Columns after renaming:")
print(df.columns.tolist())

Columns after renaming:
['name', 'miles_per_gallon', 'cylinders', 'displacement', 'hp', 'weight_lbs', 'acceleration', 'year', 'origin', 'weight_kg', 'mpg_category', 'vehicle_age']


In [13]:
#12. Filter the Data

mpg_above_30 = df[df["miles_per_gallon"] > 30]

print("Vehicles with MPG > 30:")
display(mpg_above_30)

hp_above_150 = df[df["hp"] > 150]

print("Vehicles with horsepower > 150:")
display(hp_above_150)

six_or_more_cylinders = df[df["cylinders"] >= 6]

print("Vehicles with cylinders >= 6:")
display(six_or_more_cylinders)

after_1980 = df[df["year"] > 1980]

print("Vehicles manufactured after 1980:")
display(after_1980)

usa_vehicles = df[df["origin"] == "usa"]

print("Vehicles originating from USA:")
display(usa_vehicles)

Vehicles with MPG > 30:


,name,miles_per_gallon,cylinders,displacement,hp,weight_lbs,acceleration,year,origin,weight_kg,mpg_category,vehicle_age
53,toyota corolla 1200,31.0,4.0,71.0,65.0,1773.0,19.0,71,NaN,804.218616,High,1955
54,datsun 1200,35.0,4.0,72.0,69.0,1613.0,18.0,71,NaN,731.643896,High,1955
129,datsun b210,31.0,4.0,79.0,67.0,1950.0,19.0,74,NaN,884.504400,High,1952
131,toyota corolla 1200,32.0,4.0,71.0,92.5,1836.0,15.5,74,NaN,832.794912,High,1952
144,toyota corona,31.0,4.0,76.0,52.0,1649.0,16.5,74,NaN,747.973208,High,1952
...,...,...,...,...,...,...,...,...,...,...,...,...
390,toyota celica gt,32.0,4.0,144.0,96.0,2665.0,13.9,82,NaN,1208.822680,High,1944
391,dodge charger 2.2,36.0,4.0,135.0,84.0,2370.0,13.0,82,NaN,1075.013040,High,1944
394,vw pickup,44.0,4.0,97.0,52.0,2130.0,24.6,82,NaN,966.150960,High,1944
395,dodge rampage,32.0,4.0,135.0,84.0,2295.0,11.6,82,NaN,1040.993640,High,1944


Vehicles with horsepower > 150:


,name,miles_per_gallon,cylinders,displacement,hp,weight_lbs,acceleration,year,origin,weight_kg,mpg_category,vehicle_age
1,buick skylark 320,15.0,8.0,350.0,165.0,3693.0,11.5,70,NaN,1675.115256,Low,1956
5,ford galaxie 500,15.0,8.0,429.0,198.0,4341.0,10.0,70,NaN,1969.042872,Low,1956
6,chevrolet impala,14.0,8.0,454.0,220.0,2797.5,9.0,70,NaN,1268.923620,Low,1956
7,plymouth fury iii,14.0,8.0,440.0,215.0,4312.0,8.5,70,NaN,1955.888704,Low,1956
9,amc ambassador dpl,15.0,8.0,390.0,190.0,3850.0,8.5,70,NaN,1746.329200,Low,1956
10,dodge challenger se,15.0,8.0,383.0,170.0,3563.0,10.0,70,NaN,1616.148296,Low,1956
11,plymouth 'cuda 340,14.0,8.0,340.0,160.0,3609.0,8.0,70,NaN,1637.013528,Low,1956
13,buick estate wagon (sw),14.0,8.0,455.0,225.0,3086.0,10.0,70,NaN,1399.784912,Low,1956
25,ford f250,10.0,8.0,360.0,215.0,4615.0,14.0,70,NaN,2093.327080,Low,1956
26,chevy c20,10.0,8.0,307.0,200.0,4376.0,15.0,70,NaN,1984.918592,Low,1956


Vehicles with cylinders >= 6:


,name,miles_per_gallon,cylinders,displacement,hp,weight_lbs,acceleration,year,origin,weight_kg,mpg_category,vehicle_age
0,chevrolet chevelle malibu,18.0,8.0,307.0,130.0,3504.0,12.0,70,NaN,1589.386368,Low,1956
1,buick skylark 320,15.0,8.0,350.0,165.0,3693.0,11.5,70,NaN,1675.115256,Low,1956
2,plymouth satellite,18.0,8.0,318.0,150.0,3436.0,11.0,70,NaN,1558.542112,Low,1956
3,amc rebel sst,16.0,8.0,304.0,150.0,3433.0,12.0,70,NaN,1557.181336,Low,1956
5,ford galaxie 500,15.0,8.0,429.0,198.0,4341.0,10.0,70,NaN,1969.042872,Low,1956
...,...,...,...,...,...,...,...,...,...,...,...,...
365,ford granada gl,20.2,6.0,200.0,88.0,3060.0,17.1,81,NaN,1387.991520,Medium,1945
366,chrysler lebaron salon,17.6,6.0,225.0,85.0,3465.0,16.6,81,NaN,1571.696280,Low,1945
386,buick century limited,25.0,6.0,181.0,110.0,2945.0,16.4,82,NaN,1335.828440,Medium,1944
387,oldsmobile cutlass ciera (diesel),38.0,6.0,262.0,85.0,3015.0,17.0,82,NaN,1367.579880,High,1944


Vehicles manufactured after 1980:


,name,miles_per_gallon,cylinders,displacement,hp,weight_lbs,acceleration,year,origin,weight_kg,mpg_category,vehicle_age


Vehicles originating from USA:


,name,miles_per_gallon,cylinders,displacement,hp,weight_lbs,acceleration,year,origin,weight_kg,mpg_category,vehicle_age


In [16]:
#STEP 4 — ENCODING CATEGORICAL DATA
#13. Label Encoding

from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
df["origin_encoded"] = le.fit_transform(df["origin"])

display(df[["origin", "origin_encoded"]].drop_duplicates())

mapping = dict(zip(le.classes_, le.transform(le.classes_)))

print("Category-to-label mapping:")
print(mapping)

,origin,origin_encoded
0,NaN,0


Category-to-label mapping:
{nan: np.int64(0)}


In [17]:
#14. One-Hot Encoding

one_hot_df = pd.get_dummies(
    df,
    columns=["origin"],
    prefix="origin"
)

display(one_hot_df.head())

print("Label Encoding:")
display(df[["origin", "origin_encoded"]].drop_duplicates())

print("\nOne-Hot Encoding:")
display(
    one_hot_df[
        [col for col in one_hot_df.columns if col.startswith("origin_")]
    ].drop_duplicates()
)

,name,miles_per_gallon,cylinders,displacement,hp,weight_lbs,acceleration,year,weight_kg,mpg_category,vehicle_age,origin_encoded
0,chevrolet chevelle malibu,18.0,8.0,307.0,130.0,3504.0,12.0,70,1589.386368,Low,1956,0
1,buick skylark 320,15.0,8.0,350.0,165.0,3693.0,11.5,70,1675.115256,Low,1956,0
2,plymouth satellite,18.0,8.0,318.0,150.0,3436.0,11.0,70,1558.542112,Low,1956,0
3,amc rebel sst,16.0,8.0,304.0,150.0,3433.0,12.0,70,1557.181336,Low,1956,0
4,ford torino,17.0,4.0,302.0,140.0,3449.0,10.5,70,1564.438808,Low,1956,0


Label Encoding:


,origin,origin_encoded
0,NaN,0



One-Hot Encoding:


,origin_encoded
0,0


In [18]:
#STEP 5 — OUTLIER DETECTION
#15. Identify Outliers Using Z-Scores

from scipy import stats

numerical_features = df.select_dtypes(include=np.number).columns.tolist()

print("Numerical features:")
print(numerical_features)

z_scores = np.abs(stats.zscore(df[numerical_features]))

z_score_df = pd.DataFrame(
    z_scores,
    columns=numerical_features,
    index=df.index
)

display(z_score_df.head())

outlier_counts = (z_score_df > 3).sum()

print("Number of outliers in each numerical column:")
print(outlier_counts)

outlier_mask = (z_score_df > 3).any(axis=1)

outlier_rows = df[outlier_mask]

print("Rows containing outliers:")
display(outlier_rows)

print("Total observations containing outliers:", outlier_mask.sum())

Numerical features:
['miles_per_gallon', 'cylinders', 'displacement', 'hp', 'weight_lbs', 'acceleration', 'year', 'weight_kg', 'vehicle_age', 'origin_encoded']


,miles_per_gallon,cylinders,displacement,hp,weight_lbs,acceleration,year,weight_kg,vehicle_age,origin_encoded
0,0.706439,1.515897,1.096516,0.694154,0.641001,1.301636,1.627426,0.641001,1.627426,NaN
1,1.090751,1.515897,1.510055,1.627150,0.865428,1.484357,1.627426,0.865428,1.627426,NaN
2,0.706439,1.515897,1.202305,1.227295,0.560255,1.667078,1.627426,0.560255,1.627426,NaN
3,0.962647,1.515897,1.067664,1.227295,0.556693,1.301636,1.627426,0.556693,1.627426,NaN
4,0.834543,0.847774,1.048430,0.960725,0.575692,1.849799,1.627426,0.575692,1.627426,NaN


Number of outliers in each numerical column:
miles_per_gallon    0
cylinders           0
displacement        0
hp                  4
weight_lbs          0
acceleration        2
year                0
weight_kg           0
vehicle_age         0
origin_encoded      0
dtype: int64
Rows containing outliers:


,name,miles_per_gallon,cylinders,displacement,hp,weight_lbs,acceleration,year,origin,weight_kg,mpg_category,vehicle_age,origin_encoded
6,chevrolet impala,14.0,8.0,454.0,220.0,2797.5,9.0,70,NaN,1268.923620,Low,1956,0
13,buick estate wagon (sw),14.0,8.0,455.0,225.0,3086.0,10.0,70,NaN,1399.784912,Low,1956,0
95,buick electra 225 custom,12.0,8.0,455.0,225.0,4951.0,11.0,73,NaN,2245.733992,Low,1953,0
116,pontiac grand prix,16.0,8.0,400.0,230.0,4278.0,9.5,73,NaN,1940.466576,Low,1953,0
299,peugeot 504,27.2,4.0,141.0,71.0,3190.0,24.8,79,NaN,1446.958480,Medium,1947,0
394,vw pickup,44.0,4.0,97.0,52.0,2130.0,24.6,82,NaN,966.150960,High,1944,0


Total observations containing outliers: 6


In [19]:
#STEP 6 — FEATURE SCALING
#16. Standardization Using StandardScaler

from sklearn.preprocessing import StandardScaler

scaling_features = [
    "cylinders",
    "displacement",
    "hp",
    "weight_lbs",
    "acceleration",
    "year",
    "weight_kg",
    "vehicle_age"
]

scaler = StandardScaler()

standardized_data = df.copy()

standardized_data[scaling_features] = scaler.fit_transform(
    df[scaling_features]
)

display(standardized_data[scaling_features].head())

print("Mean after standardization:")
print(standardized_data[scaling_features].mean())

print("\nStandard deviation after standardization:")
print(standardized_data[scaling_features].std())

,cylinders,displacement,hp,weight_lbs,acceleration,year,weight_kg,vehicle_age
0,1.515897,1.096516,0.694154,0.641001,-1.301636,-1.627426,0.641001,1.627426
1,1.515897,1.510055,1.627150,0.865428,-1.484357,-1.627426,0.865428,1.627426
2,1.515897,1.202305,1.227295,0.560255,-1.667078,-1.627426,0.560255,1.627426
3,1.515897,1.067664,1.227295,0.556693,-1.301636,-1.627426,0.556693,1.627426
4,-0.847774,1.048430,0.960725,0.575692,-1.849799,-1.627426,0.575692,1.627426


Mean after standardization:
cylinders      -1.963812e-16
displacement   -8.926416e-17
hp              3.570567e-17
weight_lbs     -1.249698e-16
acceleration   -6.427020e-16
year           -1.642461e-15
weight_kg       1.963812e-16
vehicle_age    -2.520820e-14
dtype: float64

Standard deviation after standardization:
cylinders       1.001259
displacement    1.001259
hp              1.001259
weight_lbs      1.001259
acceleration    1.001259
year            1.001259
weight_kg       1.001259
vehicle_age     1.001259
dtype: float64


In [20]:
#STEP 7 — NORMALIZATION
#17. Normalization Using MinMaxScaler

from sklearn.preprocessing import MinMaxScaler

minmax_scaler = MinMaxScaler()

normalized_data = df.copy()

normalized_data[scaling_features] = minmax_scaler.fit_transform(
    df[scaling_features]
)

display(normalized_data[scaling_features].head())

print("Minimum values:")
print(normalized_data[scaling_features].min())

print("\nMaximum values:")
print(normalized_data[scaling_features].max())

,cylinders,displacement,hp,weight_lbs,acceleration,year,weight_kg,vehicle_age
0,1.0,0.617571,0.456522,0.536150,0.238095,0.0,0.536150,1.0
1,1.0,0.728682,0.646739,0.589736,0.208333,0.0,0.589736,1.0
2,1.0,0.645995,0.565217,0.516870,0.178571,0.0,0.516870,1.0
3,1.0,0.609819,0.565217,0.516019,0.238095,0.0,0.516019,1.0
4,0.2,0.604651,0.510870,0.520556,0.148810,0.0,0.520556,1.0


Minimum values:
cylinders       0.0
displacement    0.0
hp              0.0
weight_lbs      0.0
acceleration    0.0
year            0.0
weight_kg       0.0
vehicle_age     0.0
dtype: float64

Maximum values:
cylinders       1.0
displacement    1.0
hp              1.0
weight_lbs      1.0
acceleration    1.0
year            1.0
weight_kg       1.0
vehicle_age     1.0
dtype: float64


In [21]:
# STEP 8 — CREATE FEATURES AND TARGET
# 18. Create X and Y Variables

# Make sure origin has meaningful labels
origin_mapping = {
    1: "usa",
    2: "europe",
    3: "japan"
}

df["origin"] = df["origin"].replace(origin_mapping)

# Create one-hot encoded columns
model_df = pd.get_dummies(
    df,
    columns=["origin"],
    prefix="origin",
    dtype=int
)

# Make sure all three origin columns exist
for col in ["origin_europe", "origin_japan", "origin_usa"]:
    if col not in model_df.columns:
        model_df[col] = 0

# Select target
Y = model_df["miles_per_gallon"].copy()

# Select input features
feature_columns = [
    "cylinders",
    "displacement",
    "hp",
    "weight_lbs",
    "acceleration",
    "year",
    "weight_kg",
    "vehicle_age",
    "origin_europe",
    "origin_japan",
    "origin_usa"
]

X = model_df[feature_columns].copy()

print("Shape of X:", X.shape)
print("Shape of Y:", Y.shape)

# Combine X and Y
automobile_X_Y = X.copy()
automobile_X_Y["miles_per_gallon"] = Y

# Display first 5 rows
display(automobile_X_Y.head())

# Save X and Y
automobile_X_Y.to_csv(
    "automobile_X_Y.csv",
    index=False
)

print("automobile_X_Y.csv saved successfully.")

# Load the CSV again
loaded_X_Y = pd.read_csv("automobile_X_Y.csv")

print("First 5 rows after loading:")
display(loaded_X_Y.head())

Shape of X: (398, 11)
Shape of Y: (398,)


,cylinders,displacement,hp,weight_lbs,acceleration,year,weight_kg,vehicle_age,origin_europe,origin_japan,origin_usa,miles_per_gallon
0,8.0,307.0,130.0,3504.0,12.0,70,1589.386368,1956,0,0,0,18.0
1,8.0,350.0,165.0,3693.0,11.5,70,1675.115256,1956,0,0,0,15.0
2,8.0,318.0,150.0,3436.0,11.0,70,1558.542112,1956,0,0,0,18.0
3,8.0,304.0,150.0,3433.0,12.0,70,1557.181336,1956,0,0,0,16.0
4,4.0,302.0,140.0,3449.0,10.5,70,1564.438808,1956,0,0,0,17.0


automobile_X_Y.csv saved successfully.
First 5 rows after loading:


,cylinders,displacement,hp,weight_lbs,acceleration,year,weight_kg,vehicle_age,origin_europe,origin_japan,origin_usa,miles_per_gallon
0,8.0,307.0,130.0,3504.0,12.0,70,1589.386368,1956,0,0,0,18.0
1,8.0,350.0,165.0,3693.0,11.5,70,1675.115256,1956,0,0,0,15.0
2,8.0,318.0,150.0,3436.0,11.0,70,1558.542112,1956,0,0,0,18.0
3,8.0,304.0,150.0,3433.0,12.0,70,1557.181336,1956,0,0,0,16.0
4,4.0,302.0,140.0,3449.0,10.5,70,1564.438808,1956,0,0,0,17.0


In [22]:
#STEP 9 — SAVE THE FINAL DATASET
#19. Save the Preprocessed Dataset

final_df = X.copy()

final_df["miles_per_gallon"] = Y

print("Final dataset:")
display(final_df)

print("Missing values in final dataset:")
print(final_df.isnull().sum())

print("Final dataset shape:", final_df.shape)

final_df.to_csv(
    "automobile_preprocessed.csv",
    index=False
)

print("automobile_preprocessed.csv saved successfully.")

Final dataset:


,cylinders,displacement,hp,weight_lbs,acceleration,year,weight_kg,vehicle_age,origin_europe,origin_japan,origin_usa,miles_per_gallon
0,8.0,307.0,130.0,3504.0,12.0,70,1589.386368,1956,0,0,0,18.0
1,8.0,350.0,165.0,3693.0,11.5,70,1675.115256,1956,0,0,0,15.0
2,8.0,318.0,150.0,3436.0,11.0,70,1558.542112,1956,0,0,0,18.0
3,8.0,304.0,150.0,3433.0,12.0,70,1557.181336,1956,0,0,0,16.0
4,4.0,302.0,140.0,3449.0,10.5,70,1564.438808,1956,0,0,0,17.0
...,...,...,...,...,...,...,...,...,...,...,...,...
393,4.0,140.0,86.0,2790.0,15.6,82,1265.521680,1944,0,0,0,27.0
394,4.0,97.0,52.0,2130.0,24.6,82,966.150960,1944,0,0,0,44.0
395,4.0,135.0,84.0,2295.0,11.6,82,1040.993640,1944,0,0,0,32.0
396,4.0,120.0,79.0,2625.0,18.6,82,1190.679000,1944,0,0,0,28.0


Missing values in final dataset:
cylinders           0
displacement        0
hp                  0
weight_lbs          0
acceleration        0
year                0
weight_kg           0
vehicle_age         0
origin_europe       0
origin_japan        0
origin_usa          0
miles_per_gallon    0
dtype: int64
Final dataset shape: (398, 12)
automobile_preprocessed.csv saved successfully.


In [23]:
import pandas as pd
import glob
import os

# Find all CSV files in the current folder
csv_files = glob.glob("*.csv")

print("CSV files found:")
print("----------------")

for file in csv_files:
    print(f"\n📄 {file}")
    
    df_temp = pd.read_csv(file)
    
    print("Shape:", df_temp.shape)
    print("Columns:", list(df_temp.columns))
    print("Missing values:")
    print(df_temp.isnull().sum())

CSV files found:
----------------

📄 Automobile.csv
Shape: (398, 9)
Columns: ['name', 'mpg', 'cylinders', 'displacement', 'horsepower', 'weight', 'acceleration', 'model_year', 'origin']
Missing values:
name             0
mpg              0
cylinders        3
displacement     3
horsepower      12
weight           2
acceleration     3
model_year       0
origin           0
dtype: int64

📄 automobile_preprocessed.csv
Shape: (398, 12)
Columns: ['cylinders', 'displacement', 'hp', 'weight_lbs', 'acceleration', 'year', 'weight_kg', 'vehicle_age', 'origin_europe', 'origin_japan', 'origin_usa', 'miles_per_gallon']
Missing values:
cylinders           0
displacement        0
hp                  0
weight_lbs          0
acceleration        0
year                0
weight_kg           0
vehicle_age         0
origin_europe       0
origin_japan        0
origin_usa          0
miles_per_gallon    0
dtype: int64

📄 automobile_X_Y.csv
Shape: (398, 12)
Columns: ['cylinders', 'displacement', 'hp', 'weight_lbs